# Aula 12 — Matrizes de incidência, adjacência e custos

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**


## Objetivo

Construir $A$, $W$ e $B$ usando a mesma ordem de vértices e IDs da Aula 11, e verificar a conservação de volume. Considera-se calda incompressível com densidade constante; vazões em L/min permitem um balanço volumétrico equivalente ao balanço de massa após multiplicação pela densidade.


## Definições e convenções

| Matriz | Dimensão no modelo básico | Definição |
|---|---|---|
| Adjacência $A$ | 11 × 11 | 1 quando o arco está disponível no modo consultado |
| Custos $W$ | 11 × 11 | Comprimento do arco, infinito sem ligação, zero na diagonal |
| Incidência $B$ | 11 × 11 | −1 na origem, +1 no destino, zero nos demais nós |

`B` descreve o **catálogo físico**; seus IDs permanecem fixos após uma falha. Em trechos indisponíveis, a vazão deve ser zero. `A` e `W` refletem a disponibilidade operacional. A diagonal zero de `W` significa permanecer no mesmo nó e não introduz uma tubulação de comprimento nulo.

Cada coluna de $B$ soma zero. Para um grafo subjacente com $c$ componentes, $\operatorname{rank}(B)=n-c$ e a dimensão do núcleo é $m-n+c$. No catálogo básico conectado: posto 10 e dimensão 1. O retorno de recirculação explica esse ciclo independente. Com o filtro reserva: $n=12$, $m=13$, dimensão 2; o segundo ciclo é do grafo **não dirigido subjacente**, sem obrigar a existência de um novo ciclo dirigido.


## Balanço com armazenamento

A equação geral adotada é:

$$\frac{d\mathbf V}{dt}=B\mathbf Q+\mathbf u-\mathbf c.$$

Aqui $B\mathbf Q$ é entrada interna menos saída interna; $\mathbf u$ é alimentação externa e $\mathbf c$ é retirada externa, ambas positivas. Para nós sem acúmulo, o lado direito vale zero. **O tanque embarcado esvazia durante o voo**, portanto não é correto impor regime permanente a ele.

No cenário nominal, e05/e06/e07 conduzem 2 L/min; e08 a e11 conduzem 0,5 L/min cada. Os trechos de solo têm vazão zero. Assim, $B\mathbf Q$ resulta em −2 L/min em `DRN_TQ_01`, +0,5 em cada bico e zero nos equipamentos intermediários. A retirada externa pelos bicos cancela esses quatro termos positivos.

Para um reservatório de 30 L, partindo de 24 L e pulverizando por 3 min, restam $24-2\times3=18$ L, ou 60%. A vazão total é medida por `DRN_FT_01`; a divisão igual entre bicos é uma hipótese do cenário.


## Exemplo de inconsistência

Se a vazão e08 for alterada isoladamente de 0,5 para 0,7 L/min, mantendo e07 em 2, o distribuidor apresenta resíduo de −0,2 L/min. A soma global continua zero por construção de $B$; isso **não prova** que todos os nós estejam balanceados. O resíduo pode indicar dados inconsistentes, acúmulo não modelado ou retirada não cadastrada; não identifica sozinho um vazamento.


## Atividades resolvidas e entregável

O notebook imprime as três matrizes, verifica as somas das colunas, calcula o balanço, simula o consumo do tanque e detecta a inconsistência. Também demonstra $BB^T=D-A_u$, em que $A_u$ usa a multiplicidade das conexões do grafo subjacente, inclusive as direções opostas. As assertivas conferem valores numéricos conhecidos e fluxos nulos fora do modo voo.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [1]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [2]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [3]:
def matriz_texto(M, linhas, colunas):
    return tabela([{'no': r, **{c: ('inf' if M[i][j] == inf else round(M[i][j], 3))
                   for j, c in enumerate(colunas)}} for i, r in enumerate(linhas)])


In [4]:
class GrafoTubulacao:
    """Dígrafo simples; catálogo físico e disponibilidade operacional separados.

    Sensores são atributos; válvulas podem estar nas arestas. Não há suposição
    de uma válvula em cada trecho. Arestas paralelas são rejeitadas.
    """
    def __init__(self, vertices):
        self.vertices = list(vertices)
        if len(set(self.vertices)) != len(self.vertices):
            raise ValueError('Vértices duplicados')
        self.arestas = {}
        self.adj = {v: [] for v in self.vertices}

    def adicionar_tubulacao(self, id_, origem, destino, comprimento_m,
                           valvula=None, diametro_mm=8.0, modos=('voo',)):
        if origem not in self.adj or destino not in self.adj:
            raise KeyError('Extremidade não cadastrada')
        if origem == destino or id_ in self.arestas:
            raise ValueError('Laço ou ID repetido')
        if any(self.arestas[e]['destino'] == destino for e in self.adj[origem]):
            raise ValueError('Aresta paralela: use um multigrafo com IDs próprios')
        if not isfinite(comprimento_m) or comprimento_m < 0 or not isfinite(diametro_mm) or diametro_mm <= 0:
            raise ValueError('Peso finito não negativo e diâmetro positivo são exigidos')
        self.arestas[id_] = dict(origem=origem, destino=destino, peso=float(comprimento_m),
                                 valvula=valvula, diametro_mm=diametro_mm,
                                 modos=tuple(modos), disponivel=True)
        self.adj[origem].append(id_)

    def vizinhos(self, u, modo=None, bloqueados=()):
        if u not in self.adj:
            raise KeyError(u)
        if u in bloqueados:
            return
        for id_ in self.adj[u]:
            a = self.arestas[id_]
            if a['disponivel'] and (modo is None or modo in a['modos']) and a['destino'] not in bloqueados:
                yield a['destino'], a['peso'], id_

    def bloquear(self, id_):
        self.arestas[id_]['disponivel'] = False

    def matrizes(self, modo=None):
        n = len(self.vertices)
        idx = {v: i for i, v in enumerate(self.vertices)}
        ids = list(self.arestas)
        A = [[0]*n for _ in range(n)]
        W = [[0.0 if i == j else inf for j in range(n)] for i in range(n)]
        B = [[0]*len(ids) for _ in range(n)]
        for j, id_ in enumerate(ids):
            a = self.arestas[id_]; u, v = idx[a['origem']], idx[a['destino']]
            # B descreve o catálogo físico, inclusive trechos indisponíveis.
            B[u][j], B[v][j] = -1, 1
            if a['disponivel'] and (modo is None or modo in a['modos']):
                A[u][v], W[u][v] = 1, a['peso']
        return A, W, B, ids

    def graus(self, modo=None):
        A, _, _, _ = self.matrizes(modo)
        return [{'no': v, 'entrada': sum(r[i] for r in A), 'saida': sum(A[i])}
                for i, v in enumerate(self.vertices)]


In [5]:
def criar_rede(redundante=False):
    nos = ['EST_TQ_01', 'EST_PMP_01', 'EST_MAN_01', 'DRN_TQ_01',
           'DRN_FLT_01', 'DRN_PMP_01', 'DRN_MAN_01'] + ['DRN_BICO_'+s for s in 'ABCD']
    if redundante:
        nos.append('DRN_FLT_02')
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao('e01', nos[0], nos[1], 1.5, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e02', nos[1], nos[2], 1.0, diametro_mm=20, modos=('mistura','abastecimento'))
    g.adicionar_tubulacao('e03', nos[2], nos[0], 2.0, 'EST_XV_REC', 20, ('mistura',))
    g.adicionar_tubulacao('e04', nos[2], nos[3], 3.0, 'EST_XV_01', 15, ('abastecimento',))
    g.adicionar_tubulacao('e05', nos[3], nos[4], 0.4, 'DRN_XV_F1_IN')
    g.adicionar_tubulacao('e06', nos[4], nos[5], 0.3, 'DRN_XV_F1_OUT')
    g.adicionar_tubulacao('e07', nos[5], nos[6], 0.5)
    for i, s in enumerate('ABCD'):
        g.adicionar_tubulacao(f'e{8+i:02}', nos[6], 'DRN_BICO_'+s, 0.6+0.2*i, 'DRN_XV_01'+s)
    if redundante:
        g.adicionar_tubulacao('e12', 'DRN_TQ_01', 'DRN_FLT_02', 0.5, 'DRN_XV_F2_IN')
        g.adicionar_tubulacao('e13', 'DRN_FLT_02', 'DRN_PMP_01', 0.6, 'DRN_XV_F2_OUT')
    return g


## Experimento 1 — Três matrizes com rótulos

In [6]:
rede = criar_rede()
A,W,B,ids = rede.matrizes('voo')
for nome,M,colunas in [('ADJACÊNCIA',A,rede.vertices),('CUSTOS (m)',W,rede.vertices),('INCIDÊNCIA FÍSICA',B,ids)]:
    print('\n'+nome+'\n'+matriz_texto(M,rede.vertices,colunas))
assert all(sum(linha[j] for linha in B)==0 for j in range(len(ids)))



ADJACÊNCIA
no         | EST_TQ_01 | EST_PMP_01 | EST_MAN_01 | DRN_TQ_01 | DRN_FLT_01 | DRN_PMP_01 | DRN_MAN_01 | DRN_BICO_A | DRN_BICO_B | DRN_BICO_C | DRN_BICO_D
-----------+-----------+------------+------------+-----------+------------+------------+------------+------------+------------+------------+-----------
EST_TQ_01  | 0         | 0          | 0          | 0         | 0          | 0          | 0          | 0          | 0          | 0          | 0         
EST_PMP_01 | 0         | 0          | 0          | 0         | 0          | 0          | 0          | 0          | 0          | 0          | 0         
EST_MAN_01 | 0         | 0          | 0          | 0         | 0          | 0          | 0          | 0          | 0          | 0          | 0         
DRN_TQ_01  | 0         | 0          | 0          | 0         | 1          | 0          | 0          | 0          | 0          | 0          | 0         
DRN_FLT_01 | 0         | 0          | 0          | 0         | 0          | 

## Experimento 2 — Balanço e consumo

In [7]:
Q = [0,0,0,0,2,2,2,0.5,0.5,0.5,0.5]
saldo = [sum(b*q for b,q in zip(linha,Q)) for linha in B]
c = [0.5 if v.startswith('DRN_BICO_') else 0 for v in rede.vertices]
acumulo = [s-retirada for s,retirada in zip(saldo,c)]
print(tabela([{'no':v,'BQ_L_min':s,'retirada_L_min':cc,'dV_dt_L_min':a}
             for v,s,cc,a in zip(rede.vertices,saldo,c,acumulo)]))
idx = {v:i for i,v in enumerate(rede.vertices)}
assert saldo[idx['DRN_TQ_01']] == -2
assert all(a==0 for v,a in zip(rede.vertices,acumulo) if v!='DRN_TQ_01')
assert sum(acumulo) == -2
assert all(q==0 for id_,q in zip(ids,Q) if 'voo' not in rede.arestas[id_]['modos'])
volume_final = 24 + acumulo[idx['DRN_TQ_01']]*3
assert volume_final == 18
print('Após 3 min:',volume_final,'L =',100*volume_final/30,'%')
Q_inconsistente = Q[:]; Q_inconsistente[7] = 0.7
residuo = sum(b*q for b,q in zip(B[idx['DRN_MAN_01']],Q_inconsistente))
assert abs(residuo+0.2)<1e-9
print('Resíduo no distribuidor:',round(residuo,3),'L/min')


no         | BQ_L_min | retirada_L_min | dV_dt_L_min
-----------+----------+----------------+------------
EST_TQ_01  | 0.0      | 0              | 0.0        
EST_PMP_01 | 0.0      | 0              | 0.0        
EST_MAN_01 | 0.0      | 0              | 0.0        
DRN_TQ_01  | -2.0     | 0              | -2.0       
DRN_FLT_01 | 0.0      | 0              | 0.0        
DRN_PMP_01 | 0.0      | 0              | 0.0        
DRN_MAN_01 | 0.0      | 0              | 0.0        
DRN_BICO_A | 0.5      | 0.5            | 0.0        
DRN_BICO_B | 0.5      | 0.5            | 0.0        
DRN_BICO_C | 0.5      | 0.5            | 0.0        
DRN_BICO_D | 0.5      | 0.5            | 0.0        
Após 3 min: 18.0 L = 60.0 %
Resíduo no distribuidor: -0.2 L/min


## Experimento 3 — Identidade laplaciana

In [8]:
n = len(rede.vertices)
BBt = [[sum(B[i][k]*B[j][k] for k in range(len(ids))) for j in range(n)] for i in range(n)]
Au = [[0]*n for _ in range(n)]
for a in rede.arestas.values():
    u,v=idx[a['origem']],idx[a['destino']]
    Au[u][v]+=1; Au[v][u]+=1
L = [[sum(Au[i]) if i==j else -Au[i][j] for j in range(n)] for i in range(n)]
assert BBt == L
print('BB^T = D - A_u confirmado, considerando multiplicidades.')


BB^T = D - A_u confirmado, considerando multiplicidades.
